# ПЗ 7 — Распознавание объектов через Gemini (Timeweb API)

Отправляем кадры на FastAPI-сервер с европейским IP, который вызывает Gemini без региональных ограничений.

In [ ]:
!pip install requests pillow tqdm -q

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

BASE_DRIVE  = '/content/drive/MyDrive/cv-frames'
FRAMES_DIR  = f'{BASE_DRIVE}/кадры'
RESULTS_DIR = f'{BASE_DRIVE}/результаты'

os.makedirs(RESULTS_DIR, exist_ok=True)

frames = sorted(f for f in os.listdir(FRAMES_DIR) if f.endswith('.jpg'))
print(f'кадров всего: {len(frames)}')


In [ ]:
import requests

API_URL = 'http://92.51.37.40:8000'

# проверяем что сервер доступен
resp = requests.get(f'{API_URL}/')
print('статус сервера:', resp.json())


In [ ]:
import json
import time
from tqdm.notebook import tqdm

def describe_frame(image_path):
    with open(image_path, 'rb') as f:
        resp = requests.post(
            f'{API_URL}/describe',
            files={'file': (os.path.basename(image_path), f, 'image/jpeg')}
        )
    resp.raise_for_status()
    return resp.json()

# каждый 10-й кадр, не больше 20 штук
STEP  = max(1, len(frames) // 20)
BATCH = frames[::STEP][:20]
PAUSE = 3  # пауза между запросами (сек)

print(f'обрабатываем {len(BATCH)} кадров')


In [ ]:
results = []

for i, fname in enumerate(tqdm(BATCH, desc='gemini-api')):
    try:
        res = describe_frame(f'{FRAMES_DIR}/{fname}')
        res['frame'] = fname
        results.append(res)
        print(f'{fname}: {res.get("objects", "")}')
    except Exception as e:
        print(f'{fname}: ошибка — {e}')
        results.append({'frame': fname, 'objects': 'ошибка', 'error': str(e)})

    if i < len(BATCH) - 1:
        time.sleep(PAUSE)

with open(f'{RESULTS_DIR}/llm_detections.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f'\nобработано: {len(results)} кадров')
print(f'сохранено: {RESULTS_DIR}/llm_detections.json')
